In [2]:
import numpy as np, pandas as pd, json, os, time, hashlib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)

for d in ['figures', 'artifacts', 'models', 'outputs']:
    os.makedirs(d, exist_ok=True)

RAW_PATH = 'data/onlineretail.csv'

In [4]:
import os
import urllib.request
import zipfile
import pandas as pd

os.makedirs('data', exist_ok=True)

zip_url = "https://archive.ics.uci.edu/static/public/352/online+retail.zip"
zip_path = "data/online_retail.zip"

if not os.path.exists("data/onlineretail.csv"):
    print("Downloading UCI Online Retail dataset...")
    urllib.request.urlretrieve(zip_url, zip_path)

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall("data/")

    xlsx_file = [f for f in os.listdir("data") if f.endswith(('.xlsx', '.xls'))][0]
    print(f"Converting {xlsx_file} to data/onlineretail.csv...")

    df_raw = pd.read_excel(os.path.join("data", xlsx_file))

    df_raw['InvoiceDate'] = pd.to_datetime(df_raw['InvoiceDate']).dt.strftime('%d.%m.%y %H:%M')
    df_raw.to_csv('data/onlineretail.csv', sep=';', decimal=',', index=False, encoding='utf-8-sig')
    print("Ready: data/onlineretail.csv created successfully.")

Converting Online Retail.xlsx to data/onlineretail.csv...
Ready: data/onlineretail.csv created successfully.


In [5]:
with open(RAW_PATH, 'rb') as f:
    file_hash = hashlib.sha256(f.read()).hexdigest()[:16]

df = pd.read_csv(RAW_PATH, sep=';', decimal=',', encoding='utf-8-sig')
raw_rows = len(df)

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], format='%d.%m.%y %H:%M')
df['is_cancel'] = df['InvoiceNo'].astype(str).str.startswith('C')

n_cancel = int(df['is_cancel'].sum())
n_missing_cust = int(df['CustomerID'].isna().sum())
n_bad_qty_price = int((((df['Quantity'] <= 0) | (df['UnitPrice'] < 0)) & (~df['is_cancel'])).sum())

clean = df[(~df['is_cancel']) & (df['Quantity'] > 0) & (df['UnitPrice'] >= 0)].copy()
clean = clean.dropna(subset=['CustomerID']).copy()
clean['CustomerID'] = clean['CustomerID'].astype(int).astype(str)
clean['StockCode'] = clean['StockCode'].astype(str)
before_dedup = len(clean)
clean = clean.drop_duplicates()
n_dupes = before_dedup - len(clean)
clean['Amount'] = clean['Quantity'] * clean['UnitPrice']
filtered_rows = len(clean)

dataset_card = {
    'source_url': 'https://archive.ics.uci.edu/dataset/352/online+retail',
    'file_sha256_16': file_hash, 'raw_rows': raw_rows, 'filtered_rows': filtered_rows,
    'rows_removed_cancellations': n_cancel, 'rows_removed_missing_customerid': n_missing_cust,
    'rows_removed_bad_qty_or_price': n_bad_qty_price, 'exact_duplicate_rows_dropped': int(n_dupes),
    'unique_customers': int(clean['CustomerID'].nunique()), 'unique_items': int(clean['StockCode'].nunique()),
    'date_min': str(clean['InvoiceDate'].min()), 'date_max': str(clean['InvoiceDate'].max()),
}
json.dump(dataset_card, open('artifacts/dataset_card.json', 'w'), indent=2, default=str)
print(json.dumps(dataset_card, indent=2, default=str))

{
  "source_url": "https://archive.ics.uci.edu/dataset/352/online+retail",
  "file_sha256_16": "4107e4dd408c0204",
  "raw_rows": 541909,
  "filtered_rows": 392732,
  "rows_removed_cancellations": 9288,
  "rows_removed_missing_customerid": 135080,
  "rows_removed_bad_qty_or_price": 1338,
  "exact_duplicate_rows_dropped": 5192,
  "unique_customers": 4339,
  "unique_items": 3665,
  "date_min": "2010-12-01 08:26:00",
  "date_max": "2011-12-09 12:50:00"
}


In [6]:
t1 = clean['InvoiceDate'].quantile(0.70)
t2 = clean['InvoiceDate'].quantile(0.85)

train_hist = clean[clean.InvoiceDate < t1].copy()
val_future = clean[(clean.InvoiceDate >= t1) & (clean.InvoiceDate < t2)].copy()
test_future = clean[clean.InvoiceDate >= t2].copy()

item_counts = train_hist.groupby('StockCode')['InvoiceNo'].nunique().sort_values(ascending=False)
CANDIDATE_SIZE = 1500
candidate_items = item_counts.head(CANDIDATE_SIZE).index.tolist()
assert 500 <= len(candidate_items) <= 2000

split_manifest = {
    't1_cutoff': str(t1), 't2_cutoff': str(t2),
    'train_hist_rows': len(train_hist), 'val_future_rows': len(val_future), 'test_future_rows': len(test_future),
    'candidate_catalog_size': len(candidate_items),
    'candidate_policy': 'Top-1500 items by unique-invoice count within train_hist',
}
json.dump(split_manifest, open('artifacts/split_manifest.json', 'w'), indent=2, default=str)
json.dump({'candidate_items': candidate_items, 'size': len(candidate_items)}, open('artifacts/candidate_policy.json', 'w'))
print(json.dumps(split_manifest, indent=2, default=str))

{
  "t1_cutoff": "2011-10-07 13:19:48",
  "t2_cutoff": "2011-11-11 12:10:00",
  "train_hist_rows": 274912,
  "val_future_rows": 58866,
  "test_future_rows": 58954,
  "candidate_catalog_size": 1500,
  "candidate_policy": "Top-1500 items by unique-invoice count within train_hist"
}


In [7]:
FEATURE_COLS = [
    'cust_txns', 'cust_items', 'cust_qty', 'cust_spend', 'cust_recency_days',
    'cust_active_days', 'cust_avg_basket', 'cust_avg_spend_per_txn', 'cust_freq_per_day',
    'item_txns', 'item_buyers', 'item_qty', 'item_avg_price', 'item_recency_days', 'item_repeat_rate',
    'pair_purchases', 'pair_qty', 'pair_spend', 'has_prior_purchase',
]

def customer_features(hist, cutoff):
    g = hist.groupby('CustomerID')
    out = g.agg(cust_txns=('InvoiceNo','nunique'), cust_items=('StockCode','nunique'),
                cust_qty=('Quantity','sum'), cust_spend=('Amount','sum'),
                cust_last=('InvoiceDate','max'), cust_first=('InvoiceDate','min')).reset_index()
    out['cust_recency_days'] = (cutoff - out['cust_last']).dt.days
    out['cust_active_days'] = (out['cust_last'] - out['cust_first']).dt.days + 1
    out['cust_avg_basket'] = out['cust_qty'] / out['cust_txns']
    out['cust_avg_spend_per_txn'] = out['cust_spend'] / out['cust_txns']
    out['cust_freq_per_day'] = out['cust_txns'] / out['cust_active_days']
    return out.drop(columns=['cust_last', 'cust_first'])

def item_features(hist, cutoff):
    g = hist.groupby('StockCode')
    out = g.agg(item_txns=('InvoiceNo','nunique'), item_buyers=('CustomerID','nunique'),
                item_qty=('Quantity','sum'), item_avg_price=('UnitPrice','mean'),
                item_last=('InvoiceDate','max'), item_first=('InvoiceDate','min')).reset_index()
    out['item_recency_days'] = (cutoff - out['item_last']).dt.days
    out['item_repeat_rate'] = out['item_txns'] / out['item_buyers']
    return out.drop(columns=['item_last', 'item_first'])

def pair_features(hist):
    return (hist.groupby(['CustomerID','StockCode'])
            .agg(pair_purchases=('InvoiceNo','nunique'), pair_qty=('Quantity','sum'),
                 pair_spend=('Amount','sum'), pair_last=('InvoiceDate','max')).reset_index())

def build_pair_table(hist, customers, candidate_items, cutoff, positives_map, n_neg, seed):
    rng = np.random.default_rng(seed)
    cf, itf, pf = customer_features(hist, cutoff), item_features(hist, cutoff), pair_features(hist)
    rows = []
    for cust in customers:
        positives = positives_map.get(cust, set())
        pos_in_cand = [i for i in positives if i in set(candidate_items)]
        for it in pos_in_cand:
            rows.append((cust, it, 1))
        pool = np.array(list(set(candidate_items) - set(pos_in_cand)))
        n = min(n_neg, len(pool))
        if n > 0:
            for it in rng.choice(pool, size=n, replace=False):
                rows.append((cust, it, 0))
    pairs = pd.DataFrame(rows, columns=['CustomerID', 'StockCode', 'label'])
    pairs = pairs.merge(cf, on='CustomerID', how='left').merge(itf, on='StockCode', how='left')
    pairs = pairs.merge(pf[['CustomerID','StockCode','pair_purchases','pair_qty','pair_spend']],
                         on=['CustomerID','StockCode'], how='left')
    for c in ['pair_purchases', 'pair_qty', 'pair_spend']:
        pairs[c] = pairs[c].fillna(0)
    pairs['has_prior_purchase'] = (pairs['pair_purchases'] > 0).astype(int)
    pairs[FEATURE_COLS] = pairs[FEATURE_COLS].fillna(0)
    return pairs

def build_scoring_table(hist, customers, candidate_items, cutoff):
    cf, itf, pf = customer_features(hist, cutoff), item_features(hist, cutoff), pair_features(hist)
    cust_df = pd.DataFrame({'CustomerID': customers}); cust_df['_k'] = 1
    item_df = pd.DataFrame({'StockCode': candidate_items}); item_df['_k'] = 1
    table = cust_df.merge(item_df, on='_k').drop(columns='_k')
    table = table.merge(cf, on='CustomerID', how='left').merge(itf, on='StockCode', how='left')
    table = table.merge(pf[['CustomerID','StockCode','pair_purchases','pair_qty','pair_spend']],
                         on=['CustomerID','StockCode'], how='left')
    for c in ['pair_purchases', 'pair_qty', 'pair_spend']:
        table[c] = table[c].fillna(0)
    table['has_prior_purchase'] = (table['pair_purchases'] > 0).astype(int)
    table[FEATURE_COLS] = table[FEATURE_COLS].fillna(0)
    return table

json.dump({'feature_cols': FEATURE_COLS}, open('artifacts/feature_schema.json', 'w'), indent=2)
print(len(FEATURE_COLS), 'features defined')

19 features defined


In [10]:
def precision_at_k(recs, relevant, k):
    recs = list(recs)[:k]
    return len(set(recs) & set(relevant)) / k if k else np.nan

def recall_at_k(recs, relevant, k):
    if not relevant: return np.nan
    return len(set(list(recs)[:k]) & set(relevant)) / len(set(relevant))

def hit_rate_at_k(recs, relevant, k):
    if not relevant: return np.nan
    return float(len(set(list(recs)[:k]) & set(relevant)) > 0)

def ndcg_at_k(recs, relevant, k):
    rel = set(relevant)
    if not rel: return np.nan
    recs = list(recs)[:k]
    dcg = sum(1.0/np.log2(i+2) for i,item in enumerate(recs) if item in rel)
    idcg = sum(1.0/np.log2(i+2) for i in range(min(len(rel), k)))
    return dcg/idcg if idcg > 0 else np.nan

def evaluate_ranking(scored_table, relevant_map, ks=(5,10,20), score_col='score',
                      user_col='CustomerID', item_col='StockCode'):
    results = {k: {'precision': [], 'recall': [], 'hitrate': [], 'ndcg': []} for k in ks}
    maxk = max(ks)
    for user, grp in scored_table.groupby(user_col):
        relevant = relevant_map.get(user, set())
        if not relevant: continue
        recs = grp.sort_values(score_col, ascending=False)[item_col].tolist()[:maxk]
        for k in ks:
            results[k]['precision'].append(precision_at_k(recs, relevant, k))
            results[k]['recall'].append(recall_at_k(recs, relevant, k))
            results[k]['hitrate'].append(hit_rate_at_k(recs, relevant, k))
            results[k]['ndcg'].append(ndcg_at_k(recs, relevant, k))
    return {k: {m: float(np.nanmean(v)) for m, v in results[k].items()} for k in ks}

In [8]:
train_customers_all = sorted(train_hist['CustomerID'].unique().tolist())
val_pos = (val_future.groupby('CustomerID')['StockCode'].apply(lambda s: set(s.unique()))).to_dict()

cand_set = set(candidate_items)
users_with_future = [c for c in train_customers_all if len(val_pos.get(c, set())) > 0]
recall_num = sum(len(val_pos[c] & cand_set) for c in users_with_future)
recall_den = sum(len(val_pos[c]) for c in users_with_future)
candidate_recall_val = recall_num / recall_den
print('candidate recall (val window):', round(candidate_recall_val, 4))

pairs = build_pair_table(train_hist, train_customers_all, candidate_items, t1, val_pos, n_neg=50, seed=SEED)

rng = np.random.default_rng(SEED)
shuffled = rng.permutation(train_customers_all)
n_val = int(0.2 * len(shuffled))
val_customers = set(shuffled[:n_val].tolist())
train_customers = set(shuffled[n_val:].tolist())
pairs_train = pairs[pairs.CustomerID.isin(train_customers)].reset_index(drop=True)
pairs_val = pairs[pairs.CustomerID.isin(val_customers)].reset_index(drop=True)
print(pairs_train.shape, pairs_val.shape, 'positives:', pairs_train.label.sum(), pairs_val.label.sum())

candidate recall (val window): 0.794
(173337, 22) (43962, 22) positives: 25187 6962


In [11]:
val_customers_with_future = [c for c in sorted(pairs_val.CustomerID.unique()) if len(val_pos.get(c, set())) > 0]
sample_n = min(300, len(val_customers_with_future))
sample_customers = list(rng.choice(val_customers_with_future, size=sample_n, replace=False))
val_score_table = build_scoring_table(train_hist, sample_customers, candidate_items, t1)

grid = [
    dict(n_estimators=200, max_depth=None, min_samples_leaf=2, max_features='sqrt'),
    dict(n_estimators=300, max_depth=None, min_samples_leaf=2, max_features='sqrt'),
    dict(n_estimators=300, max_depth=12,   min_samples_leaf=5, max_features='sqrt'),
]
X_train, y_train = pairs_train[FEATURE_COLS], pairs_train['label']
X_val, y_val = pairs_val[FEATURE_COLS], pairs_val['label']

tuning_rows = []
for params in grid:
    rf = RandomForestClassifier(**params, class_weight='balanced_subsample', random_state=SEED, n_jobs=-1)
    rf.fit(X_train, y_train)
    val_proba = rf.predict_proba(X_val)[:, 1]
    scored = val_score_table.copy()
    scored['score'] = rf.predict_proba(scored[FEATURE_COLS])[:, 1]
    rank = evaluate_ranking(scored, val_pos, ks=(10,))
    tuning_rows.append({**params, 'roc_auc': roc_auc_score(y_val, val_proba),
                         'pr_auc': average_precision_score(y_val, val_proba),
                         'val_recall_10': rank[10]['recall'], 'val_ndcg_10': rank[10]['ndcg']})
    print(tuning_rows[-1])

tuning_df = pd.DataFrame(tuning_rows)
tuning_df.to_csv('artifacts/hyperparam_validation.csv', index=False)
best_params = grid[tuning_df['val_recall_10'].idxmax()]
print('SELECTED:', best_params)

{'n_estimators': 200, 'max_depth': None, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'roc_auc': np.float64(0.8741522201604075), 'pr_auc': np.float64(0.6661668492538723), 'val_recall_10': 0.1486424569322766, 'val_ndcg_10': 0.328180656716568}
{'n_estimators': 300, 'max_depth': None, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'roc_auc': np.float64(0.8755030881930479), 'pr_auc': np.float64(0.6673916317876023), 'val_recall_10': 0.14967385179414058, 'val_ndcg_10': 0.3380950406646322}


KeyboardInterrupt: 

In [ ]:
dev_hist = pd.concat([train_hist, val_future], ignore_index=True)
dev_customers_all = sorted(dev_hist['CustomerID'].unique().tolist())
test_pos = (test_future.groupby('CustomerID')['StockCode'].apply(lambda s: set(s.unique()))).to_dict()

users_with_future_test = [c for c in dev_customers_all if len(test_pos.get(c, set())) > 0]
recall_num = sum(len(test_pos[c] & cand_set) for c in users_with_future_test)
recall_den = sum(len(test_pos[c]) for c in users_with_future_test)
candidate_recall_test = recall_num / recall_den
print('candidate recall (locked test window):', round(candidate_recall_test, 4))

dev_pairs = build_pair_table(dev_hist, dev_customers_all, candidate_items, t2, test_pos, n_neg=50, seed=SEED)

rf = RandomForestClassifier(**best_params, class_weight='balanced_subsample', random_state=SEED, n_jobs=-1)
rf.fit(dev_pairs[FEATURE_COLS], dev_pairs['label'])
joblib.dump(rf, 'models/random_forest.joblib')

importance = pd.Series(rf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
importance.to_csv('artifacts/feature_importance.csv', header=['importance'])
print(importance)

In [ ]:
all_test_customers = [c for c in test_pos if len(test_pos[c]) > 0]
test_table = build_scoring_table(dev_hist, all_test_customers, candidate_items, t2)
test_table['rf_score'] = rf.predict_proba(test_table[FEATURE_COLS])[:, 1]

pop_counts = dev_hist.groupby('StockCode')['InvoiceNo'].nunique()
test_table['pop_score'] = test_table['StockCode'].map(pop_counts).fillna(0)

dev_customers_set = set(dev_hist['CustomerID'].unique())
test_table['is_warm'] = test_table['CustomerID'].isin(dev_customers_set)
warm_customers = [c for c in all_test_customers if c in dev_customers_set]
print('warm customers:', len(warm_customers), '/ all:', len(all_test_customers))

In [ ]:
warm_sub = test_table[test_table.CustomerID.isin(warm_customers)]
rf_result = evaluate_ranking(warm_sub, test_pos, ks=(5,10,20), score_col='rf_score')
pop_result = evaluate_ranking(warm_sub, test_pos, ks=(5,10,20), score_col='pop_score')

metrics_df = pd.DataFrame([
    {'Model': 'Popularity', **{f'{m}@{k}': pop_result[k][m] for k in (5,10) for m in ('precision','recall','hitrate')}},
    {'Model': 'Random Forest', **{f'{m}@{k}': rf_result[k][m] for k in (5,10) for m in ('precision','recall','hitrate')}},
])
metrics_df.to_csv('outputs/23MID0328_Lab07_Ranking_Metrics.csv', index=False)
metrics_df

In [ ]:
GRAY = ['#1a1a1a', '#4d4d4d', '#808080', '#b3b3b3', '#d9d9d9']
plt.rcParams.update({'font.size': 10})

# Top-15 items
top15 = train_hist.groupby('StockCode')['InvoiceNo'].nunique().sort_values(ascending=False).head(15)
plt.figure(figsize=(7,4))
plt.barh(range(len(top15)), top15.values[::-1], color=GRAY[1])
plt.yticks(range(len(top15)), top15.index[::-1])
plt.title('Top 15 Items by Transaction Count'); plt.xlabel('Unique invoices')
plt.tight_layout(); plt.savefig('figures/top15_items.png', dpi=150); plt.show()

# Feature importance
plt.figure(figsize=(7,5))
imp_sorted = importance.sort_values()
plt.barh(range(len(imp_sorted)), imp_sorted.values, color=GRAY[1])
plt.yticks(range(len(imp_sorted)), imp_sorted.index)
plt.title('Random Forest Feature Importance'); plt.xlabel('Gini importance')
plt.tight_layout(); plt.savefig('figures/feature_importance.png', dpi=150); plt.show()

# Popularity vs RF @10
metrics_names = ['precision','recall','hitrate']
x = np.arange(len(metrics_names)); w = 0.35
plt.figure(figsize=(6,4))
plt.bar(x-w/2, [pop_result[10][m] for m in metrics_names], width=w, label='Popularity', color=GRAY[3], edgecolor='black')
plt.bar(x+w/2, [rf_result[10][m] for m in metrics_names], width=w, label='Random Forest', color=GRAY[0], edgecolor='black')
plt.xticks(x, ['Precision@10','Recall@10','HitRate@10']); plt.legend()
plt.title('Popularity vs Random Forest (K=10)')
plt.tight_layout(); plt.savefig('figures/pop_vs_rf.png', dpi=150); plt.show()

In [ ]:
candidate_recall_df = pd.DataFrame([
    {'window': 'train_hist -> val_future', 'candidate_recall': round(candidate_recall_val, 4),
     'n_users_with_future_positives': len(users_with_future)},
    {'window': 'dev_hist -> test_future (locked)', 'candidate_recall': round(candidate_recall_test, 4),
     'n_users_with_future_positives': len(users_with_future_test)},
])
candidate_recall_df.to_csv('outputs/23MID0328_Lab07_Candidate_Recall.csv', index=False)

cold_customers = set(all_test_customers) - dev_customers_set
cold_start_summary = {'n_cold_start_customers': len(cold_customers), 'n_warm_customers': len(warm_customers),
                       'pct_cold_start': round(len(cold_customers)/len(all_test_customers), 4)}
print(cold_start_summary)

rf_top10 = (warm_sub.sort_values(['CustomerID','rf_score'], ascending=[True,False])
            .groupby('CustomerID').head(10))
rf_top10[['CustomerID','StockCode','rf_score']].to_csv('outputs/23MID0328_Lab07_Recommendations.csv', index=False)

rf_top5 = (warm_sub.sort_values(['CustomerID','rf_score'], ascending=[True,False])
           .groupby('CustomerID')['StockCode'].apply(lambda s: s.tolist()[:5]))
pop_top5 = (warm_sub.sort_values(['CustomerID','pop_score'], ascending=[True,False])
            .groupby('CustomerID')['StockCode'].apply(lambda s: s.tolist()[:5]))

records = []
hist_size = dev_hist.groupby('CustomerID')['InvoiceNo'].nunique()
for c in warm_customers:
    rel = test_pos[c]
    rf_recs, pop_recs = rf_top5.get(c, []), pop_top5.get(c, [])
    records.append({'cust': c, 'hist_size': int(hist_size.get(c,0)), 'n_relevant': len(rel),
                     'rf_hits': len(set(rf_recs)&rel), 'pop_hits': len(set(pop_recs)&rel)})
error_df = pd.DataFrame(records)
error_df.to_csv('outputs/23MID0328_Lab07_Error_Analysis.csv', index=False)
error_df.sort_values('rf_hits', ascending=False).head(5)